<img src="imagens/banner.png" alt="Módulo 05: Paginação e navegação" width="1100">

# Módulo 05: Paginação e navegação

Sites raramente mostram tudo em uma página só. Um catálogo de mil livros fica dividido em 50 páginas, e os detalhes de cada livro ficam em uma página própria. Para coletar um site inteiro, o scraper precisa **andar por ele**: de uma página para a seguinte, e de uma listagem para cada item.

Você já fez isso uma vez, na missão do Módulo 03, trocando o número no endereço até o servidor responder `404`. Este módulo apresenta essa e outras formas de percorrer um site, e quando usar cada uma.

<img src="imagens/mapa-do-curso.png" alt="Mapa dos 10 módulos do curso, com o Módulo 05 destacado" width="1100">

## Neste módulo você vai

- Encontrar o **padrão da URL** de um site e montar o endereço de cada página;
- Descobrir **como cada site avisa** que as páginas acabaram: `404`, página vazia, ausência do botão "próximo";
- **Seguir o botão "próximo"**, de página em página, até o fim;
- Completar **links relativos** com o `urljoin`;
- Ler a **lista de páginas** que o site mostra e percorrer todas;
- Navegar **em profundidade**: da página inicial às categorias, das categorias aos livros;
- Coletar os detalhes de **uma categoria inteira** e salvar o resultado em um arquivo.

**Tempo estimado:** de 2h a 2h30.

## Como este módulo funciona

Como nos Módulos 03 e 04: explicações curtas, células curtas, e exercícios **Sua vez** divididos em passos, com as respostas ocultas. Vários exemplos fazem dezenas de pedidos, sempre com pausas entre eles: algumas células levam de 10 a 30 segundos.

**Antes de começar:** abra o ambiente do curso (rotina do Módulo 01) e abra este notebook, `05-paginacao-e-navegacao.ipynb`.

▶️ Importe as bibliotecas. O `urljoin` e o `json` são novos e serão explicados quando forem usados.

In [ ]:
import json
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
from parsel import Selector

▶️ Todos os pedidos do módulo passam por uma única **sessão**, com um `User-Agent` que identifica o scraper, como no Módulo 03. A função `pedir()` faz o pedido e já corrige a codificação:

In [ ]:
sessao = requests.Session()
sessao.headers["User-Agent"] = "curso-web-scraping (aluno@exemplo.com)"

def pedir(url, **opcoes):
    resposta = sessao.get(url, timeout=20, **opcoes)
    resposta.encoding = "utf-8"
    return resposta

---

# Parte 1: Montando a URL de cada página

## 1.1 Encontrando o padrão

A forma mais simples de percorrer um site é a que você já usou: descobrir como o endereço muda de uma página para outra e **montar** cada endereço com código. Para descobrir o padrão, abra o site no navegador, vá para as páginas 2 e 3 e compare a barra de endereço:

<img src="imagens/padroes-url.png" alt="Endereços das páginas 1, 2 e 3 de quatro sites de treino, com o número que muda destacado em amarelo, e o modelo de cada um" width="1120">

Cada site escreve o número de um jeito, mas a ideia é a mesma: uma parte fixa e um pedaço que muda.

▶️ Monte os endereços das três primeiras páginas do catálogo com uma f-string. Ainda não há nenhum pedido: só os textos.

In [ ]:
for numero in range(1, 4):
    print(f"https://books.toscrape.com/catalogue/page-{numero}.html")

▶️ Agora peça cada página e veja o status. A pausa (`time.sleep`) mantém um ritmo responsável:

In [ ]:
for numero in range(1, 4):
    resposta = pedir(f"https://books.toscrape.com/catalogue/page-{numero}.html")
    print(numero, resposta.status_code)
    time.sleep(0.5)

## 1.2 Até o 404

No Módulo 03, o laço pedia página após página até o servidor responder `404`. É a forma mais simples de saber onde o catálogo acaba, e funciona em muitos sites. Para não baixar as 50 páginas de novo, o exemplo abaixo começa na página 48.

▶️ O `while True` repete até o `break`, que acontece no primeiro `404`:

In [ ]:
numero = 48
while True:
    resposta = pedir(f"https://books.toscrape.com/catalogue/page-{numero}.html")
    print(numero, resposta.status_code)
    if resposta.status_code == 404:
        break
    numero += 1
    time.sleep(0.5)

A página 51 não existe, e o laço parou ali. Mas nem todo site responde `404` depois da última página.

## 1.3 Quando não há 404

Antes de montar o laço, descubra **como o site avisa** que as páginas acabaram:

<img src="imagens/condicoes-parada.png" alt="Quatro sinais de fim: o código 404, a página com 200 mas sem itens, a ausência do botão próximo e o total informado na página" width="1120">

A loja de exemplo do webscraper.io tem 4 páginas de tablets. Veja o que acontece na página 5.

▶️ Peça a página 5. O número vai no parâmetro `page`:

In [ ]:
url_tablets = "https://webscraper.io/test-sites/e-commerce/static/computers/tablets"
resposta = pedir(url_tablets, params={"page": 5})
resposta.status_code

O status é `200`: para o servidor, a página existe.

▶️ Conte os produtos que ela tem. Cada produto fica em um contêiner `div` cuja classe contém `thumbnail`:

In [ ]:
pagina = Selector(text=resposta.text)
len(pagina.xpath('//div[contains(@class, "thumbnail")]'))

Nenhum produto: a página existe, mas está **vazia**. Um laço que esperasse o `404` nunca pararia aqui. O sinal de fim, neste site, é a página sem itens.

▶️ Crie uma função que pede uma página de uma categoria da loja e devolve quantos produtos ela tem:

In [ ]:
def contar_produtos(categoria, numero):
    url = f"https://webscraper.io/test-sites/e-commerce/static/{categoria}"
    pagina = Selector(text=pedir(url, params={"page": numero}).text)
    return len(pagina.xpath('//div[contains(@class, "thumbnail")]'))

▶️ Teste com as páginas 4 e 5 dos tablets:

In [ ]:
print(contar_produtos("computers/tablets", 4), contar_produtos("computers/tablets", 5))

A página 4 tem 3 produtos, e a 5, nenhum.

▶️ Agora o laço, que para na primeira página vazia. Por segurança, ele também tem um **limite máximo** de páginas: se o sinal de fim falhar, o scraper não fica pedindo páginas para sempre.

In [ ]:
numero, total, limite = 1, 0, 50
while numero <= limite:
    produtos = contar_produtos("computers/tablets", numero)
    if produtos == 0:
        break
    total += produtos
    numero += 1
    time.sleep(0.5)
print(f"{numero - 1} páginas, {total} produtos")

São 4 páginas e 21 tablets. A linha `numero, total, limite = 1, 0, 50` cria as três variáveis de uma vez.

### Sua vez: todas as páginas de notebooks

**Passo 1 de 2.** Use `contar_produtos()` para ver quantos produtos tem a página 1 da categoria `computers/laptops`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
contar_produtos("computers/laptops", 1)
```
</details>

**Passo 2 de 2.** Adapte o laço da seção 1.3 para a categoria `computers/laptops` e descubra quantas páginas e quantos notebooks ela tem. O laço leva cerca de 20 segundos.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
numero, total, limite = 1, 0, 50
while numero <= limite:
    produtos = contar_produtos("computers/laptops", numero)
    if produtos == 0:
        break
    total += produtos
    numero += 1
    time.sleep(0.5)
print(f"{numero - 1} páginas, {total} produtos")
```
</details>

São 20 páginas e 117 notebooks. O mesmo laço serve para qualquer categoria da loja: só o nome muda.

---

# Parte 2: Seguindo o botão "próximo"

## 2.1 Encontrando o botão

Quase toda listagem com várias páginas tem um botão para a **próxima página**. Em vez de montar o endereço, o scraper pode ler o link desse botão e segui-lo. Use o que você aprendeu no Módulo 04 para encontrá-lo: o seletor de elementos e a busca do Inspetor.

Na página 1 do catálogo, o botão **next** fica no fim da página:

<img src="imagens/proximo-firefox.png" alt="O Firefox na página 1 do catálogo: o bloco Page 1 of 50 com o botão next destacado, a busca do Inspetor com o XPath do botão e a contagem 1 of 1, e a linha a href page-2.html marcada" width="1130">

- ① O bloco de paginação, no fim da página: o texto *Page 1 of 50* e o botão **next**.
- ② A busca do Inspetor com o XPath do botão, `//li[@class="next"]/a`: **1 resultado**, o último galho.
- ③ O elemento encontrado: `<a href="page-2.html">next</a>`. O endereço da próxima página está no atributo `href`.

▶️ Peça a página 1 e leia o `href` do botão:

In [ ]:
resposta = pedir("https://books.toscrape.com/catalogue/page-1.html")
pagina = Selector(text=resposta.text)

proximo = pagina.xpath('//li[@class="next"]/a/@href').get()
proximo

O valor é `page-2.html`: não começa com `https://`. É um **link relativo**, escrito em relação à página onde ele está, como um caminho de pastas.

▶️ Veja o que acontece ao entregar esse texto direto ao requests. Vai aparecer um erro, **de propósito**:

In [ ]:
pedir(proximo)

O erro é `MissingSchema`: falta o protocolo, e o requests não sabe para qual servidor ir. É o mesmo que tentar abrir `page-2.html` na barra de endereço do navegador, sem o site na frente.

## 2.2 Completando links relativos com urljoin

O navegador completa os links relativos sozinho, usando o endereço da página atual. No Python, quem faz isso é a função `urljoin()`, do módulo `urllib.parse`: ela recebe o endereço da página onde o link estava e o `href`, e devolve o endereço completo. Ela resolve todas as formas de link relativo:

<img src="imagens/urls-relativas.png" alt="Tabela com quatro tipos de link: mesma pasta, pastas acima com ../, a partir da raiz com barra e absoluto, cada um com a página de origem, o href e o endereço completo resolvido pelo urljoin" width="1120">

▶️ Complete o link do botão. O primeiro argumento é o endereço da página atual, que fica em `resposta.url`:

In [ ]:
urljoin(resposta.url, proximo)

▶️ Teste as quatro formas da tabela. Repare que cada `../` sobe uma pasta:

In [ ]:
pagina_categoria = "https://books.toscrape.com/catalogue/category/books/mystery_3/index.html"

print(urljoin(pagina_categoria, "page-2.html"))
print(urljoin(pagina_categoria, "../../../sharp-objects_997/index.html"))
print(urljoin("https://quotes.toscrape.com/", "/page/2/"))
print(urljoin(pagina_categoria, "https://quotes.toscrape.com/page/3/"))

> 💡 Use sempre `resposta.url` como base, e não o endereço que você pediu. Se houve um redirecionamento (Módulo 03), a página está em outro endereço, e os links relativos são contados a partir dele.

## 2.3 O laço "enquanto houver próximo"

Com o `urljoin`, o laço fica simples: peça a página, extraia os dados, procure o botão "próximo" e, se ele existir, siga para o endereço dele. Na última página o botão não existe, o `get()` devolve `None`, e o laço termina.

O exemplo percorre o quotes.toscrape.com, cujo botão "next" tem um link a partir da raiz (`/page/2/`).

▶️ Execute o laço. Ele mostra cada endereço visitado e conta as citações:

In [ ]:
url = "https://quotes.toscrape.com/"
citacoes = 0
while url:
    resposta = pedir(url)
    pagina = Selector(text=resposta.text)
    citacoes += len(pagina.xpath('//div[@class="quote"]'))
    print(resposta.url)
    proximo = pagina.xpath('//li[@class="next"]/a/@href').get()
    url = urljoin(resposta.url, proximo) if proximo else None
    time.sleep(0.5)
print(citacoes, "citações")

O laço visitou as 10 páginas e parou sozinho, sem precisar saber quantas eram. A linha `url = urljoin(...) if proximo else None` é um `if` em uma linha só: se existe um próximo, `url` recebe o endereço completo; se não, recebe `None`, e o `while url:` termina.

### Sua vez: seguindo o próximo em uma categoria

**Passo 1 de 3.** A categoria *Fantasy* do books.toscrape.com fica em `https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html`. Peça essa página e conte quantos livros (`//article[@class="product_pod"]`) ela tem.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = pedir("https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html")
pagina = Selector(text=resposta.text)
len(pagina.xpath('//article[@class="product_pod"]'))
```
</details>

**Passo 2 de 3.** Leia o `href` do botão "next" dessa página e complete-o com o `urljoin`. Para onde ele leva?

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
proximo = pagina.xpath('//li[@class="next"]/a/@href').get()
urljoin(resposta.url, proximo)
```
</details>

**Passo 3 de 3.** Escreva o laço completo, partindo da primeira página da categoria, e conte os livros de todas as páginas.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
url = "https://books.toscrape.com/catalogue/category/books/fantasy_19/index.html"
livros = 0
while url:
    resposta = pedir(url)
    pagina = Selector(text=resposta.text)
    livros += len(pagina.xpath('//article[@class="product_pod"]'))
    proximo = pagina.xpath('//li[@class="next"]/a/@href').get()
    url = urljoin(resposta.url, proximo) if proximo else None
    time.sleep(0.5)
livros
```
</details>

São 48 livros em 3 páginas. Repare que o `href` do botão é `page-2.html` também aqui, mas o `urljoin` o completa dentro da pasta da categoria, porque a base é a página atual.

---

# Parte 3: Lendo a lista de páginas

## 3.1 O total que o site mostra

Muitos sites mostram uma lista com o número de cada página, ou um texto com o total, como *Page 1 of 50*. Com o total em mãos, um `for` com `range()` percorre todas as páginas, e você sabe de antemão quanto falta.

A página de times de hóquei do scrapethissite mostra a lista completa, de 1 a 24:

<img src="imagens/lista-paginas-firefox.png" alt="O Firefox na página de times de hóquei: a lista de páginas de 1 a 24 destacada, a busca do Inspetor com o XPath dos links e a contagem 1 of 25, e o primeiro link marcado" width="1130">

- ① A lista de páginas, de 1 a 24, mais o botão `»`.
- ② A busca do Inspetor com o XPath dos links: **25 resultados**, os 24 números e o `»`.
- ③ O primeiro link: `href="/pages/forms/?page_num=1"`.

▶️ Peça a primeira página e encontre os links da lista:

In [ ]:
resposta = pedir("https://www.scrapethissite.com/pages/forms/")
pagina = Selector(text=resposta.text)

links = pagina.xpath('//ul[@class="pagination"]//a')
len(links)

▶️ Veja o texto e o endereço dos três últimos links. O texto vem com espaços, por isso o `normalize-space()`:

In [ ]:
for link in links[-3:]:
    print(link.xpath("normalize-space(.)").get(), link.xpath("@href").get())

Repare no último: o botão `»` ("próxima") aponta para `page_num=1`, a própria página 1, e não para a 2. É um **erro do site**. Sites reais também têm erros, e por isso vale conferir o que um link realmente faz antes de confiar nele.

Para descobrir o total de páginas, o mais seguro é pegar o **maior número** entre os textos dos links.

▶️ O método `isdigit()` diz se um texto é formado só por algarismos, e o `int()` o transforma em número:

In [ ]:
numeros = []
for link in links:
    texto = link.xpath("normalize-space(.)").get()
    if texto.isdigit():
        numeros.append(int(texto))
ultima = max(numeros)
ultima

▶️ Com o total, o `range()` gera o número de cada página. O laço pede todas e conta os times. Leva cerca de 20 segundos:

In [ ]:
times = 0
for numero in range(1, ultima + 1):
    resposta = pedir("https://www.scrapethissite.com/pages/forms/", params={"page_num": numero})
    pagina = Selector(text=resposta.text)
    times += len(pagina.xpath('//tr[@class="team"]'))
    print(f"página {numero} de {ultima}", end="\r")
    time.sleep(0.3)
print(f"\n{times} linhas de times")

São 582 linhas de times. Como o total era conhecido, o laço mostrou o progresso (`página 7 de 24`), atualizando a mesma linha: é o que faz o `end="\r"`.

### Sua vez: o total escrito na página

**Passo 1 de 3.** A página 1 do catálogo do books.toscrape.com mostra o texto *Page 1 of 50*, em um `li` com a classe `current`. Peça a página e leia esse texto, limpo com `normalize-space()`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = pedir("https://books.toscrape.com/catalogue/page-1.html")
pagina = Selector(text=resposta.text)
texto = pagina.xpath('normalize-space(//li[@class="current"])').get()
texto
```
</details>

**Passo 2 de 3.** O total é a última palavra do texto. Use o `split()` e o `int()` para guardar o número em `total_paginas`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
total_paginas = int(texto.split()[-1])
total_paginas
```
</details>

**Passo 3 de 3.** Monte, **sem fazer pedidos**, a lista com os endereços de todas as páginas do catálogo, e mostre quantos são e qual é o último.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
enderecos = []
for numero in range(1, total_paginas + 1):
    enderecos.append(f"https://books.toscrape.com/catalogue/page-{numero}.html")
print(len(enderecos), enderecos[-1])
```
</details>

## 3.2 Qual estratégia escolher

As três formas chegam ao mesmo resultado. A escolha depende do que o site oferece:

<img src="imagens/estrategias.png" alt="Comparação das três estratégias: montar a URL, seguir o próximo e ler a lista de páginas, com um trecho de código, os pontos a favor e contra e quando usar cada uma" width="1120">

Na dúvida, prefira **seguir o "próximo"**: ele não depende do formato do endereço, não precisa saber o total e para sozinho na última página. Montar a URL continua sendo a forma mais rápida de testar um site e de pedir uma página específica.

---

# Parte 4: Navegando em profundidade

## 4.1 Da página inicial a cada livro

A listagem mostra só um resumo de cada livro. O código do produto (UPC), a quantidade exata em estoque, a categoria e a descrição ficam na **página do livro**. Para coletá-los, o scraper precisa descer pelos níveis do site, seguindo os links como um visitante:

<img src="imagens/mapa-navegacao.png" alt="Os níveis do books.toscrape.com: página inicial com as categorias, categoria com os livros, próximas páginas da categoria e página do livro, com o XPath dos links de cada nível" width="1120">

A missão desta parte é coletar os detalhes de **todos os livros da categoria Mystery**, descendo os três níveis.

## 4.2 Nível 1: as categorias

▶️ Peça a página inicial e encontre os links das categorias, na barra lateral:

In [ ]:
resposta = pedir("https://books.toscrape.com/")
inicial = Selector(text=resposta.text)

categorias = inicial.xpath('//div[@class="side_categories"]//ul/li/ul/li/a')
len(categorias)

▶️ Veja o nome e o endereço completo das três primeiras:

In [ ]:
for categoria in categorias[:3]:
    nome = categoria.xpath("normalize-space(.)").get()
    print(nome, urljoin(resposta.url, categoria.xpath("@href").get()))

▶️ Encontre o link da categoria *Mystery* pelo texto do link. Como o texto tem espaços em volta, a condição usa `normalize-space()`:

In [ ]:
href = inicial.xpath('//div[@class="side_categories"]//a[normalize-space()="Mystery"]/@href').get()
url_mystery = urljoin(resposta.url, href)
url_mystery

## 4.3 Nível 2: os livros da categoria

▶️ Peça a primeira página da categoria e veja o link do primeiro livro:

In [ ]:
resposta = pedir(url_mystery)
pagina = Selector(text=resposta.text)
pagina.xpath('//article[@class="product_pod"]//h3/a/@href').get()

O link começa com `../../../`: ele sobe três pastas a partir da pasta da categoria.

▶️ Complete-o com o `urljoin`:

In [ ]:
urljoin(resposta.url, "../../../sharp-objects_997/index.html")

▶️ Junte o que você viu em uma função: ela segue o botão "próximo" por todas as páginas de uma categoria e devolve a lista com o endereço completo de cada livro:

In [ ]:
def links_da_categoria(url):
    links = []
    while url:
        resposta = pedir(url)
        pagina = Selector(text=resposta.text)
        for href in pagina.xpath('//article[@class="product_pod"]//h3/a/@href').getall():
            links.append(urljoin(resposta.url, href))
        proximo = pagina.xpath('//li[@class="next"]/a/@href').get()
        url = urljoin(resposta.url, proximo) if proximo else None
        time.sleep(0.5)
    return links

▶️ Use a função na categoria *Mystery*:

In [ ]:
links_mystery = links_da_categoria(url_mystery)
len(links_mystery)

▶️ Veja os dois primeiros endereços:

In [ ]:
links_mystery[:2]

## 4.4 Nível 3: a página de cada livro

Na página do livro, os dados são extraídos com o que você aprendeu no Módulo 04: o título e o preço ficam no bloco `product_main`, e o UPC e a disponibilidade, na tabela de rótulo e valor.

▶️ Crie a função que pede a página de um livro e devolve um dicionário com os dados:

In [ ]:
def extrair_detalhes(url):
    pagina = Selector(text=pedir(url).text)
    principal = pagina.xpath('//div[contains(@class, "product_main")]')
    return {
        "titulo": principal.xpath("./h1/text()").get(),
        "preco": principal.xpath('./p[@class="price_color"]/text()').get(),
        "nota": principal.xpath('./p[contains(@class, "star-rating")]/@class').get().split()[-1],
        "estoque": pagina.xpath('//th[text()="Availability"]/following-sibling::td/text()').get(),
        "upc": pagina.xpath('//th[text()="UPC"]/following-sibling::td/text()').get(),
        "categoria": pagina.xpath('//ul[@class="breadcrumb"]/li[3]/a/text()').get(),
        "descricao": pagina.xpath('//div[@id="product_description"]/following-sibling::p/text()').get(),
        "url": url,
    }

▶️ Teste com o primeiro livro da lista:

In [ ]:
extrair_detalhes(links_mystery[0])

Repare no estoque: na listagem aparecia só *In stock*; aqui aparece a quantidade, *In stock (20 available)*. E a categoria e a descrição só existem nesta página.

## 4.5 Missão: a categoria inteira

▶️ Percorra os 32 links, extraindo os detalhes de cada livro. O `enumerate()` numera os itens, para mostrar o progresso. Leva cerca de 30 segundos:

In [ ]:
livros_mystery = []
for i, link in enumerate(links_mystery, 1):
    livros_mystery.append(extrair_detalhes(link))
    print(f"{i} de {len(links_mystery)}", end="\r")
    time.sleep(0.5)
len(livros_mystery)

▶️ Confira o último livro coletado:

In [ ]:
livros_mystery[-1]

Para que os dados não se percam quando o notebook for fechado, eles vão para um arquivo. O formato **JSON** (Módulo 03) guarda listas e dicionários como texto, e o módulo `json` faz a conversão: `json.dumps()` transforma a lista em texto. O `ensure_ascii=False` mantém os acentos e o `£` legíveis, e o `indent=2` deixa o arquivo organizado.

▶️ Salve a lista em `dados/mystery.json`:

In [ ]:
Path("dados").mkdir(exist_ok=True)
arquivo = Path("dados/mystery.json")
arquivo.write_text(json.dumps(livros_mystery, ensure_ascii=False, indent=2), encoding="utf-8")

O número que aparece é a quantidade de caracteres gravados.

▶️ Veja o começo do arquivo:

In [ ]:
print(arquivo.read_text(encoding="utf-8")[:400])

A categoria inteira está salva: 32 livros, cada um com 8 campos, coletados em três níveis de navegação. No Módulo 06, esses dados vão virar uma tabela, com os preços e as notas convertidos em números.

### Sua vez: em profundidade no quotes.toscrape.com

**Passo 1 de 4.** Cada citação do quotes.toscrape.com tem um link `(about)` para a página do autor. Peça a página inicial e pegue todos os links de autor: os `a` cujo `href` contém `/author/`.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
resposta = pedir("https://quotes.toscrape.com/")
pagina = Selector(text=resposta.text)
hrefs = pagina.xpath('//div[@class="quote"]//a[contains(@href, "/author/")]/@href').getall()
len(hrefs)
```
</details>

**Passo 2 de 4.** Alguns autores aparecem mais de uma vez. Crie a lista `autores`, com os endereços completos e **sem repetição**: só acrescente um endereço se ele ainda não estiver na lista (`if url not in autores`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
autores = []
for href in hrefs:
    url = urljoin(resposta.url, href)
    if url not in autores:
        autores.append(url)
len(autores)
```
</details>

**Passo 3 de 4.** Peça a página do primeiro autor e mostre o nome (`h3` com a classe `author-title`), a data de nascimento (`span` com a classe `author-born-date`) e o local (`span` com a classe `author-born-location`).

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
autor = Selector(text=pedir(autores[0]).text)
print(autor.xpath('normalize-space(//h3[@class="author-title"])').get())
print(autor.xpath('//span[@class="author-born-date"]/text()').get())
print(autor.xpath('//span[@class="author-born-location"]/text()').get())
```
</details>

**Passo 4 de 4.** Percorra todos os autores da lista, mostrando o nome e a data de nascimento de cada um, com uma pausa entre os pedidos.

In [ ]:
# Escreva o seu código aqui

<details>
<summary>Ver a resposta</summary>

```python
for url in autores:
    autor = Selector(text=pedir(url).text)
    nome = autor.xpath('normalize-space(//h3[@class="author-title"])').get()
    print(nome, "-", autor.xpath('//span[@class="author-born-date"]/text()').get())
    time.sleep(0.5)
```
</details>

A página inicial tem 10 citações, mas só 8 autores diferentes. Evitar visitar o mesmo endereço duas vezes economiza pedidos, para você e para o site.

---

# Parte 5: Navegando com responsabilidade

Um scraper que navega faz muitos pedidos. A missão deste módulo fez 34 pedidos para uma categoria de 32 livros; o catálogo inteiro, com os detalhes, passaria de mil. Algumas práticas mantêm a coleta responsável e o scraper confiável:

| Prática | Por quê | Neste módulo |
|---|---|---|
| **Pausa entre os pedidos** | Não sobrecarregar o servidor | `time.sleep(0.5)` em todos os laços |
| **Uma sessão, com `User-Agent`** | Reaproveitar a conexão e identificar o scraper | A função `pedir()` |
| **Limite máximo de páginas** | Não ficar preso se o sinal de fim falhar | `while numero <= limite` (seção 1.3) |
| **Não repetir endereços** | Economizar pedidos | A lista de autores sem repetição |
| **Salvar o resultado** | Não precisar coletar tudo de novo | `dados/mystery.json` |
| **Respeitar o `robots.txt`** | Seguir as regras do site (Módulo 02) | Os sites de treino permitem as páginas usadas |

> 💡 Quando a coleta for longa, salve o progresso aos poucos (a cada página, por exemplo). Se a internet cair no meio, você continua de onde parou. O Módulo 07 volta a esse assunto.

▶️ No fim do trabalho, feche a sessão:

In [ ]:
sessao.close()

---

## Problemas comuns e como resolver

| Sintoma | Causa provável | Solução |
|---|---|---|
| `MissingSchema: Invalid URL 'page-2.html'` | O link é relativo e foi passado direto ao requests | Complete com `urljoin(resposta.url, href)` (seção 2.2) |
| O `urljoin` gera um endereço que dá `404` | A base usada não é a página onde o link estava | Use sempre o `resposta.url` da página onde o link foi encontrado |
| O laço não termina | O site não responde `404` no fim, ou o botão "próximo" nunca some | Confira como o site avisa o fim (seção 1.3) e use um limite máximo de páginas |
| O laço termina cedo demais | O XPath do botão "próximo" ou dos itens está errado | Teste o XPath na busca do Inspetor, na última página e numa do meio |
| Os mesmos itens aparecem repetidos | Um link leva de volta a uma página já visitada (como o `»` do scrapethissite) | Guarde os endereços visitados e confira antes de pedir |
| `AttributeError: 'NoneType' object has no attribute 'split'` | Uma página de livro não tem o elemento esperado | Teste a função `extrair_detalhes()` sozinha com o endereço que falhou |
| `ValueError: max() arg is an empty sequence` | Nenhum link da lista de páginas tinha só algarismos | Confira o XPath da lista e o texto dos links |
| `NameError: name 'pedir' is not defined` | As células do início não foram executadas | Execute as células do início do notebook, em ordem |

---

# Retomada do Módulo 05

<img src="imagens/retomada.png" alt="Mapa de revisão do módulo: montar a URL, seguir o próximo, lista de páginas, links relativos, navegação em profundidade e responsabilidade" width="1100">

### O que você aprendeu

- **Montar a URL**: descubra o pedaço do endereço que muda e gere cada página com uma f-string ou com `params`. É a forma mais simples e permite pedir qualquer página direto.
- **O sinal de fim** varia de site para site: `404`, página `200` sem itens, ausência do botão "próximo" ou o total escrito na página. Coloque sempre um **limite máximo** no laço.
- **Seguir o "próximo"**: leia o `href` do botão, complete com o `urljoin` e repita enquanto ele existir. É a forma mais robusta.
- **Links relativos** (`page-2.html`, `/page/2/`, `../../../livro/`) são completados com `urljoin(resposta.url, href)`.
- **A lista de páginas**: leia o maior número, ou o total do texto, e percorra com `range()`. Confira os links: sites também têm erros.
- **Em profundidade**: da página inicial às categorias, das categorias aos itens, e dos itens aos detalhes. Cada nível é um XPath de links mais um `urljoin`.
- O `json` guarda listas e dicionários em um arquivo, para que a coleta não precise ser repetida.

### Teste seus conhecimentos

Clique em cada pergunta para ver a resposta.

<details>
<summary><b>1.</b> Um site responde <code>200</code> para qualquer número de página, mesmo depois da última. Como o laço sabe quando parar?</summary>

Contando os itens de cada página: quando vier uma página sem itens, o laço para. Por segurança, ele também deve ter um limite máximo de páginas.
</details>

<details>
<summary><b>2.</b> Qual é o resultado de <code>urljoin("https://site.com/loja/categoria/index.html", "pagina-2.html")</code>?</summary>

`https://site.com/loja/categoria/pagina-2.html`: o link relativo troca o último pedaço do caminho, como um arquivo da mesma pasta.
</details>

<details>
<summary><b>3.</b> E de <code>urljoin("https://site.com/loja/categoria/index.html", "/produtos/1")</code>?</summary>

`https://site.com/produtos/1`: o link que começa com `/` parte da raiz do site, e não da pasta atual.
</details>

<details>
<summary><b>4.</b> Por que seguir o botão "próximo" costuma ser mais robusto que montar a URL?</summary>

Porque não depende do formato do endereço nem de saber o total de páginas, e termina sozinho quando o botão some, na última página.
</details>

<details>
<summary><b>5.</b> Por que a missão visitou a página de cada livro, se a listagem já mostrava título e preço?</summary>

Porque alguns dados só existem na página do livro: o UPC, a quantidade exata em estoque, a categoria e a descrição.
</details>

### Glossário

| Termo | Significado |
|---|---|
| **Paginação** | A divisão de uma listagem em várias páginas |
| **Padrão de URL** | A parte do endereço que muda de uma página para outra |
| **Sinal de fim** | O que indica que as páginas acabaram: `404`, página vazia, falta do botão "próximo" |
| **Link relativo** | Um `href` escrito em relação à página atual, sem o protocolo e o domínio |
| **Link absoluto** | Um endereço completo, com `https://` e o domínio |
| **`urljoin()`** | A função que completa um link relativo a partir do endereço da página |
| **Raspagem em profundidade** (*crawling*) | Navegar de uma listagem para as páginas de cada item, e assim por diante |
| **JSON** | Formato de texto que guarda listas e dicionários, usado para salvar os dados coletados |

---

## Próximo módulo

Os dados coletados até aqui ainda são **texto**: o preço é `"£47.82"`, a nota é `"Four"`, o estoque é `"In stock (20 available)"`. O **Módulo 06: Do HTML ao dataset** transforma esses textos em números, organiza tudo em uma tabela com o **pandas** e salva o resultado em CSV e em Excel: o primeiro dataset do curso.

## Referências

Materiais consultados na elaboração deste módulo, agrupados pela parte em que foram usados.

**URLs e links relativos** (Partes 1 e 2)

- Documentação do Python: [urllib.parse.urljoin](https://docs.python.org/pt-br/3/library/urllib.parse.html#urllib.parse.urljoin).
- MDN Web Docs: [What is a URL?](https://developer.mozilla.org/en-US/docs/Learn_web_development/Howto/Web_mechanics/What_is_a_URL), seção sobre URLs absolutas e relativas (em inglês).

**requests e parsel** (todas as partes)

- [Session Objects](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) e [Passing Parameters In URLs](https://requests.readthedocs.io/en/latest/user/quickstart/#passing-parameters-in-urls), na documentação do requests (em inglês).
- [Usage](https://parsel.readthedocs.io/en/latest/usage.html), na documentação do parsel (em inglês).

**Salvando os dados** (Parte 4)

- Documentação do Python: [json](https://docs.python.org/pt-br/3/library/json.html).

**Leitura complementar**

- [Following links](https://docs.scrapy.org/en/latest/intro/tutorial.html#following-links), no tutorial do Scrapy (em inglês): a mesma ideia de seguir o botão "próximo", com a ferramenta do Módulo 10.

**Sites usados nos exemplos:** [books.toscrape.com](https://books.toscrape.com), [quotes.toscrape.com](https://quotes.toscrape.com), [scrapethissite.com](https://www.scrapethissite.com/pages/forms/) e a [loja de exemplo do webscraper.io](https://webscraper.io/test-sites/e-commerce/static).